# Hypothesis Testing for EDA: Solution

A worked reference for the taxi-ride exercises. Each exercise follows the guide's recipe: plot, quantify, test, interpret.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats

df = sns.load_dataset("taxis")
df.head()

---
## Exercise 1: Do Longer Rides Cost More?

Numeric vs numeric: `distance` and `fare`, so the tools are a scatter plot and a correlation test.

In [ ]:
sns.scatterplot(data=df, x="distance", y="fare", alpha=0.3)
plt.title("Fare vs Distance")
plt.show()

r, p_pearson = stats.pearsonr(df["distance"], df["fare"])
rs, p_spearman = stats.spearmanr(df["distance"], df["fare"])
print(f"Pearson r:  {r:.3f}, p-value: {p_pearson:.1e}")
print(f"Spearman r: {rs:.3f}, p-value: {p_spearman:.1e}")

**Explanation**

Pearson gives r = 0.92 and Spearman 0.93, both with p-values of essentially zero: an extremely strong, real relationship. The agreement between the two coefficients says the relationship is close to linear and not driven by outliers. Statistically this is the cleanest result in the notebook, but for a stakeholder it is a sanity check rather than a finding: taxi meters charge by distance and time, so fares must grow with distance. A "significant" result is not automatically an interesting one.

---
## Exercise 2: Do Card Payers Tip More Than Cash Payers?

Categorical (2 groups) vs numeric. `payment` has 44 missing values, so we drop those rows and state it.

In [ ]:
paid = df.dropna(subset=["payment"])
card = paid[paid["payment"] == "credit card"]["tip"]
cash = paid[paid["payment"] == "cash"]["tip"]

sns.boxplot(data=paid, x="payment", y="tip")
plt.title("Tip by Payment Method")
plt.show()

print(f"Card: n {len(card)}, mean tip {card.mean():.2f}")
print(f"Cash: n {len(cash)}, mean tip {cash.mean():.2f}")

t_stat, p_value = stats.ttest_ind(card, cash, equal_var=False)
print(f"Welch t-statistic: {t_stat:.1f}, p-value: {p_value:.1e}")

# The step that saves the analysis: look at the raw cash tips
print(f"Share of cash rides with a recorded tip of exactly 0: {(cash == 0).mean():.3f}")

**Explanation**

The t-test is as emphatic as a test can be (t = 75.6, p essentially zero): card rides average a 2.78 tip, cash rides 0.00. But the last line exposes the trap: **every single cash ride has a recorded tip of exactly zero.** NYC taxi meters only record tips paid through the card terminal; cash tips go straight into the driver's pocket and never enter the data. The pattern is a data-recording artifact, not a behavioural difference, and no hypothesis test can detect that: a test checks whether a difference is due to chance, not whether your measurement means what you think it means. The honest report: "tips are only observed for card payments; cash tipping behaviour is unmeasurable in this data".

### Exercise 2b (Trickier): Do Green Cabs Charge Higher Fares?

In [ ]:
yellow = df[df["color"] == "yellow"]["fare"]
green = df[df["color"] == "green"]["fare"]

print(
    f"Yellow: n {len(yellow)}, mean {yellow.mean():.2f}, median {yellow.median():.2f}"
)
print(f"Green:  n {len(green)}, mean {green.mean():.2f}, median {green.median():.2f}")

t_stat, p_welch = stats.ttest_ind(yellow, green, equal_var=False)
u_stat, p_mw = stats.mannwhitneyu(yellow, green)
print(f"Welch t-test p-value:   {p_welch:.4f}")
print(f"Mann-Whitney U p-value: {p_mw:.4f}")

**Explanation**

The means differ (12.92 vs 14.04, Welch p = 0.010) but the medians are identical (9.50 both), and Mann-Whitney does not reject (p = 0.061). The two tests answer different questions: Welch compares means, which are pulled by the long right tail of expensive rides; Mann-Whitney compares the distributions by rank, where the typical ride looks the same for both colours. So the extra green-cab revenue lives in the tail (green cabs work the outer boroughs, where rides are longer, as Exercise 4 confirms), not in the typical fare. The honest report: a borderline lead about tail behaviour, not "green cabs are more expensive". Picking whichever test crossed 0.05 would be p-hacking.

---
## Exercise 3: Does the Fare Depend on the Pickup Borough?

Categorical (4 groups) vs numeric: one-way ANOVA. `pickup_borough` has 26 missing values, dropped and stated.

In [ ]:
boro = df.dropna(subset=["pickup_borough"])

sns.boxplot(data=boro, x="pickup_borough", y="fare")
plt.title("Fare by Pickup Borough")
plt.show()

print(boro.groupby("pickup_borough")["fare"].agg(["count", "mean"]).round(2))

groups = [g["fare"].values for _, g in boro.groupby("pickup_borough")]
f_stat, p_value = stats.f_oneway(*groups)
print(f"ANOVA F-statistic: {f_stat:.1f}, p-value: {p_value:.1e}")

**Explanation**

ANOVA rejects overwhelmingly (F = 371, p about 2e-222): mean fares range from 11.15 in Manhattan to 24.93 in Queens. The result makes sense: Manhattan pickups are short hops, while Queens pickups include airport runs. ANOVA only says "at least one borough differs"; to name the differing pairs you would run a post-hoc test with a built-in multiple-comparison correction, such as Tukey's HSD (`statsmodels.stats.multicomp.pairwise_tukeyhsd`), never six uncorrected pairwise t-tests.

---
## Exercise 4: Do Yellow and Green Cabs Work Different Boroughs?

Categorical vs categorical: chi-square test of independence plus Cramér's V as the effect size.

In [ ]:
def cramers_v(table):
    chi2, _, _, _ = stats.chi2_contingency(table)
    n = table.to_numpy().sum()
    return np.sqrt(chi2 / (n * (min(table.shape) - 1)))


sns.countplot(data=boro, x="pickup_borough", hue="color")
plt.title("Cab Colour by Pickup Borough")
plt.show()

table_color = pd.crosstab(boro["color"], boro["pickup_borough"])
print(table_color)
chi2, p_value, dof, _ = stats.chi2_contingency(table_color)
print(
    f"Chi-square: {chi2:.1f}, p-value: {p_value:.1e}, dof: {dof}, Cramér's V: {cramers_v(table_color):.3f}"
)

# Second pair: payment vs colour
table_pay = pd.crosstab(paid["payment"], paid["color"])
chi2b, p_b, _, _ = stats.chi2_contingency(table_pay)
print(
    f"payment vs color: chi-square {chi2b:.1f}, p-value {p_b:.1e}, Cramér's V: {cramers_v(table_pay):.3f}"
)

**Explanation**

Colour and borough are strongly associated: chi-square p is essentially zero and Cramér's V = 0.62, a large effect. The table shows why: 94% of yellow pickups are in Manhattan, while green pickups spread across Brooklyn, Queens, and the Bronx. This is NYC regulation made visible in data: green (boro) cabs are licensed for street hails outside the Manhattan core.

The contrast with `payment` vs `color` is the point of the comparison: also "significant" (p about 4e-21) but with V = 0.12, a weak association. With 6,400 rides, even small dependencies clear the significance bar, so the effect size is what separates a structural pattern from a statistical footnote.

---
## Exercise 5: Reflection, the Multiple-Testing Ledger

**1. How worried should you be?**

This notebook ran about eight tests (two correlations, two two-group tests plus their robustness checks, one ANOVA, two chi-squares). On pure noise, each has a 5% false-positive chance, so the probability of at least one false alarm across eight tests is roughly 1 - 0.95^8, about 34%. A Bonferroni correction would demand p < 0.006 per test.

**2. The verdict on each finding:**

- **Solid**: fare vs distance (r = 0.92, p about 0), fare by borough (p about 2e-222), colour by borough (p about 0, V = 0.62). All three survive any correction and carry large effect sizes.
- **Lead needing confirmation**: yellow vs green fares. Welch p = 0.010 does not survive Bonferroni, Mann-Whitney disagrees, and the medians are identical: report as a tail-behaviour hypothesis, not a conclusion.
- **Artifact**: card vs cash tips. The p-value is as small as they come, and the finding is still wrong as a behavioural claim, because cash tips are simply not recorded. No correction fixes a measurement problem.
- **Significant but weak**: payment vs colour (V = 0.12). Real, but too small to matter for most decisions; report the effect size alongside the p-value.

---
## Summary

| Question | Test | p-value | Effect size | Verdict |
|---|---|---|---|---|
| Longer rides cost more? | Pearson correlation | about 0 | r = 0.92 | Real (and unsurprising) |
| Card payers tip more? | Welch t-test | about 0 | huge | Recording artifact |
| Green cabs pricier? | Welch + Mann-Whitney | 0.010 / 0.061 | means differ, medians equal | Borderline tail-effect lead |
| Fare depends on borough? | One-way ANOVA | 2e-222 | 11 to 25 mean range | Real |
| Colour tied to borough? | Chi-square | about 0 | V = 0.62 | Real and strong |
| Payment tied to colour? | Chi-square | 4e-21 | V = 0.12 | Significant but weak |